In [ ]:
# Install required libraries
!pip install ultralytics easyocr opencv-python-headless pandas -q

import os
import cv2
import torch
import easyocr
import sqlite3
from datetime import datetime
from ultralytics import YOLO
from google.colab import drive
import matplotlib.pyplot as plt

# Mount drive
drive.mount('/content/drive')

# Define core paths
PROJECT_ROOT = '/content/drive/MyDrive/MultiCamera_Vehicle_ReIdentification'
YOLO_MODEL_PATH = os.path.join(PROJECT_ROOT, 'yolo_runs', 'license_plate_detector', 'weights', 'best.pt')
DB_PATH = os.path.join(PROJECT_ROOT, 'vehicle_logs.db')

print("✅ Setup complete!")

In [ ]:
# 1. Load the trained YOLOv8 "Eyes"
print("Loading YOLOv8 Detector...")
detector = YOLO(YOLO_MODEL_PATH)

# 2. Load the EasyOCR "Brain"
print("Loading EasyOCR Reader...")
reader = easyocr.Reader(['en'], gpu=True)

# 3. Connect to the SQLite "Memory"
conn = sqlite3.connect(DB_PATH, timeout=15)
cursor = conn.cursor()

# Ensure the table exists
cursor.execute('''
CREATE TABLE IF NOT EXISTS vehicle_tracking (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
    license_plate TEXT NOT NULL,
    camera_id TEXT NOT NULL,
    timestamp DATETIME NOT NULL,
    ocr_confidence REAL
)
''')
conn.commit()

print("✅ All systems loaded and ready!")

In [ ]:
import time
import sqlite3
from datetime import datetime
import cv2

# --- SHORT TERM MEMORY BUFFER ---
recently_logged_plates = {}
COOLDOWN_SECONDS = 10 # Ignore identical plates for 10 seconds

def log_to_db(plate_text, camera_id, ocr_conf):
    """Safely opens, writes, and closes the database to prevent locking."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    try:
        # The 'with' statement guarantees the connection closes safely even if it crashes
        with sqlite3.connect(DB_PATH, timeout=20) as conn:
            cursor = conn.cursor()
            cursor.execute('''
                INSERT INTO vehicle_tracking (license_plate, camera_id, timestamp, ocr_confidence)
                VALUES (?, ?, ?, ?)
            ''', (plate_text, camera_id, timestamp, ocr_conf))
            # No need for conn.commit() or conn.close(), the 'with' block handles it!
        return True
    except sqlite3.OperationalError as e:
        print(f"⚠️ DB Locked, skipping log for {plate_text}. Error: {e}")
        return False

def preprocess_plate(plate_crop):
    """Cleans up the cropped image for better OCR accuracy."""
    gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    blur = cv2.bilateralFilter(gray, 11, 17, 17)
    return blur

def process_frame(frame, camera_id="Cam_01"):
    """Runs the full pipeline with crash-proof database logging."""
    results = detector(frame, verbose=False)[0]

    for box in results.boxes:
        x_min, y_min, x_max, y_max = [int(val) for val in box.xyxy[0]]
        confidence = float(box.conf[0])

        # 1. YOLO THRESHOLD
        if confidence > 0.6:
            plate_crop = frame[y_min:y_max, x_min:x_max]

            if plate_crop.size == 0:
                continue

            processed_crop = preprocess_plate(plate_crop)
            ocr_results = reader.readtext(processed_crop, allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789')

            if ocr_results:
                best_result = max(ocr_results, key=lambda x: x[2])
                plate_text = best_result[1]
                ocr_conf = best_result[2]

                # 2. OCR THRESHOLD
                if ocr_conf > 0.5 and len(plate_text) > 6:

                    current_time_sec = time.time()

                    # 3. REPETITION FILTER (DEBOUNCING)
                    if plate_text not in recently_logged_plates or (current_time_sec - recently_logged_plates[plate_text]) > COOLDOWN_SECONDS:

                        # 4. CRASH-PROOF LOGGING
                        success = log_to_db(plate_text, camera_id, ocr_conf)

                        if success:
                            recently_logged_plates[plate_text] = current_time_sec

                            cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (0, 255, 0), 3)
                            label = f"LOGGED: {plate_text} ({ocr_conf:.2f})"
                            cv2.putText(frame, label, (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
                            print(f"✅ LOGGED TO DB: {label} on {camera_id}")

                    else:
                        cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (0, 255, 255), 2)
                        label = f"TRACKING: {plate_text}"
                        cv2.putText(frame, label, (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    return frame

print("✅ Crash-Proof Processing engine ready!")

In [ ]:
def run_camera_stream(input_filename, output_filename, camera_name):
    """Processes an entire video file pretending to be a specific CCTV camera."""
    input_path = os.path.join(PROJECT_ROOT, input_filename)
    output_path = os.path.join(PROJECT_ROOT, 'outputs', output_filename)

    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print(f"❌ Error: Could not open {input_filename}")
        return

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0
    print(f"🎥 {camera_name} is now online and processing...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Process every 3rd frame to save time
        if frame_count % 3 == 0:
            annotated_frame = process_frame(frame, camera_id=camera_name)
        else:
            annotated_frame = frame

        out.write(annotated_frame)
        frame_count += 1

    cap.release()
    out.release()
    print(f"✅ {camera_name} feed complete! Saved to outputs folder.\n")

# --- EXECUTE THE MULTI-CAMERA NETWORK ---

# 1. Start Camera A
run_camera_stream("cam_A_video.mp4", "cam_A_tracked.mp4", "North_Gate_Cam")

# 2. Start Camera B
run_camera_stream("cam_B_video.mp4", "cam_B_tracked.mp4", "Library_Parking_Cam")

In [ ]:
import sqlite3
import pandas as pd

DB_PATH = '/content/drive/MyDrive/MultiCamera_Vehicle_ReIdentification/vehicle_logs.db'

# 1. Wipe the database clean
with sqlite3.connect(DB_PATH, timeout=10) as conn:
    cursor = conn.cursor()
    cursor.execute('DELETE FROM vehicle_tracking')
    conn.commit()

    # Verify it worked
    cursor.execute('SELECT COUNT(*) FROM vehicle_tracking')
    count = cursor.fetchone()[0]
    print(f"🧹 Database wiped clean! Current logs: {count}")

In [ ]:
import pandas as pd

# 1. Connect to the database
conn = sqlite3.connect(DB_PATH)

# 2. Query the entire tracking log, ordered by the most recent sightings
query = """
SELECT timestamp, camera_id, license_plate, ocr_confidence
FROM vehicle_tracking
ORDER BY timestamp DESC
"""

# 3. Load into a Pandas DataFrame for a clean, readable table
df = pd.read_sql_query(query, conn)

if df.empty:
    print("📭 The database is empty. No vehicles have been logged yet.")
else:
    print(f"🗺️ Total Vehicle Sightings Logged: {len(df)}")
    display(df) # This prints a beautiful table directly in Colab

In [ ]:
# 2. Extract and View the Data
with sqlite3.connect(DB_PATH, timeout=10) as conn:
    # Query all records, ordered by time to show the journey
    query = """
    SELECT timestamp, camera_id, license_plate, ocr_confidence
    FROM vehicle_tracking
    ORDER BY timestamp ASC
    """

    df = pd.read_sql_query(query, conn)

    if df.empty:
        print("📭 The database is empty.")
    else:
        print(f"🗺️ Clean Tracking Log (Total Entries: {len(df)})")
        display(df)